In [1]:
import pandas as pd
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [2]:
# Загружаем данные (предполагается, что у вас есть файл train.json)
with open('train.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Превращаем списки ингредиентов в строку
df['ingredients_str'] = df['ingredients'].apply(lambda x: ' '.join(x))

# Кодируем названия кухонь в числа
le = LabelEncoder()
y = le.fit_transform(df['cuisine'])


In [3]:
# Создаем матрицу признаков
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['ingredients_str'])

# Делим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [4]:
# Создаем и обучаем модель
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [5]:
# Делаем предсказания
predictions = model.predict(X_test)

# Выводим отчет о точности для каждого типа кухни
print(classification_report(y_test, predictions, target_names=le.classes_))


              precision    recall  f1-score   support

   brazilian       0.76      0.50      0.60        84
     british       0.65      0.32      0.43       157
cajun_creole       0.82      0.66      0.73       328
     chinese       0.75      0.88      0.81       510
    filipino       0.73      0.49      0.58       136
      french       0.60      0.66      0.63       550
       greek       0.80      0.63      0.70       249
      indian       0.87      0.90      0.89       602
       irish       0.62      0.38      0.48       151
     italian       0.78      0.91      0.84      1567
    jamaican       0.89      0.56      0.69        91
    japanese       0.85      0.68      0.76       284
      korean       0.84      0.72      0.78       166
     mexican       0.90      0.93      0.91      1336
    moroccan       0.88      0.71      0.79       166
     russian       0.68      0.36      0.47        89
 southern_us       0.65      0.80      0.72       848
     spanish       0.65    

In [6]:

# 2. Метод опорных векторов (LinearSVC)
# Обычно работает быстрее и точнее логистической регрессии на разреженных матрицах
svm_model = LinearSVC(max_iter=1000, random_state=42)
svm_model.fit(X_train, y_train)

svm_predictions = svm_model.predict(X_test)
print("--- LinearSVC ---")
print(classification_report(y_test, svm_predictions, target_names=le.classes_))


# 3. Случайный лес (RandomForestClassifier)
# Хорошо справляется со сложными нелинейными взаимосвязями
# n_jobs=-1 использует все ядра процессора для ускорения
rf_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
print("--- Random Forest ---")
print(classification_report(y_test, rf_predictions, target_names=le.classes_))


--- LinearSVC ---
              precision    recall  f1-score   support

   brazilian       0.76      0.62      0.68        84
     british       0.57      0.39      0.46       157
cajun_creole       0.78      0.65      0.71       328
     chinese       0.77      0.86      0.81       510
    filipino       0.61      0.54      0.57       136
      french       0.61      0.65      0.63       550
       greek       0.72      0.67      0.70       249
      indian       0.87      0.88      0.87       602
       irish       0.58      0.44      0.50       151
     italian       0.81      0.88      0.84      1567
    jamaican       0.79      0.69      0.74        91
    japanese       0.81      0.73      0.77       284
      korean       0.83      0.78      0.80       166
     mexican       0.91      0.93      0.92      1336
    moroccan       0.84      0.71      0.77       166
     russian       0.54      0.43      0.48        89
 southern_us       0.70      0.78      0.73       848
     span

### 📊 Сравнение базовых моделей (Логистическая регрессия, LinearSVC, Random Forest)

#### 🏆 1. Логистическая регрессия — Лучший баланс (Победитель)
* **🟢 Плюсы:**
  * Показывает самый высокий и стабильный F1-score на крупных классах (например, *italian*: `0.84`, *mexican*: `0.91`, *indian*: `0.89`).
  * Имеет отличный баланс между точностью (`precision`) и полнотой (`recall`).
* **🔴 Минусы:**
  * Хуже находит редкие классы с маленьким `support` (*british*, *russian*), выдавая по ним низкий `recall`.

#### 🥈 2. LinearSVC — Лучший выбор для редких классов (2-е место)
* **🟢 Плюсы:**
  * Значительно поднимает полноту (`recall`) на редких кухнях.
  * Для *jamaican* `recall` вырос с `0.56` до `0.69`, для *brazilian* — с `0.50` до `0.62`.
  * За счет этого имеет самый высокий общий **Macro Avg F1 (`0.69`)**.
* **🔴 Минусы:**
  * Слегка просела точность (`precision`) на средних классах по сравнению с логистической регрессией.

#### ❌ 3. Random Forest — Худший результат (Не подходит)
* **🟢 Плюсы:**
  * Высокий `precision` на некоторых отдельных классах.
* **🔴 Минусы:**
  * Модель **сильно переобучилась** под частые классы и фактически игнорирует редкие.
  * Полнота (`recall`) для *british* упала до `0.20`, для *russian* — до `0.24`, для *irish* — до `0.26`.
  * Это стандартная проблема деревьев решений на текстовых разреженных матрицах (TF-IDF с 5000 признаков).


In [9]:
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Загрузка данных
with open("train.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df["ingredients_str"] = df["ingredients"].apply(lambda x: " ".join(x))

# 2. Кодирование и векторизация
le = LabelEncoder()
y = le.fit_transform(df["cuisine"])

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["ingredients_str"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Настройка сетки параметров для базовой модели
# Мы берем базовую модель (без class_weight='balanced') и ищем лучший баланс
base_model = LogisticRegression(max_iter=1000, random_state=42)
param_grid = {"C": [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]}

# Запускаем поиск (cv=3 для скорости, n_jobs=-1 для параллельности)
grid_search = GridSearchCV(
    base_model, param_grid, cv=3, scoring="f1_weighted", n_jobs=-1
)
grid_search.fit(X_train, y_train)

# 4. Результаты
print(f"Лучшее значение параметра C: {grid_search.best_params_['C']}\n")

# Оцениваем лучшую модель на тестовой выборке
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

print("--- Logistic Regression (Optimized C) ---")
print(classification_report(y_test, predictions, target_names=le.classes_))


Лучшее значение параметра C: 5.0

--- Logistic Regression (Optimized C) ---
              precision    recall  f1-score   support

   brazilian       0.77      0.60      0.67        84
     british       0.58      0.39      0.47       157
cajun_creole       0.79      0.66      0.72       328
     chinese       0.78      0.86      0.82       510
    filipino       0.65      0.52      0.58       136
      french       0.60      0.67      0.64       550
       greek       0.75      0.67      0.71       249
      indian       0.89      0.89      0.89       602
       irish       0.61      0.46      0.52       151
     italian       0.80      0.89      0.84      1567
    jamaican       0.79      0.63      0.70        91
    japanese       0.82      0.73      0.77       284
      korean       0.82      0.75      0.79       166
     mexican       0.91      0.93      0.92      1336
    moroccan       0.89      0.71      0.79       166
     russian       0.61      0.40      0.49        89
 sout

> 📌 **Главные выводы по оптимизированной модели (C = 5.0):**
>
> * **Рост общего качества (Macro F1 = 0.70):**
>   * Это лучший результат за все итерации.
>   * Рост на 2% означает стабильное распознавание категорий по всему датасету.
>
> * **Мягкое улучшение редких кухонь:**
>   * Удалось поднять полноту (`recall`), сохранив точность (`precision`).
>   * **Британская:** F1-score вырос с **0.43** (база) до **0.47** (сейчас).
>   * **Ирландская:** F1-score вырос с **0.48** (база) до **0.52** (сейчас).
>   * **Бразильская:** F1-score вырос с **0.60** (база) до **0.67** (сейчас).
>   * **Русская:** `recall` поднялся с **0.36** до **0.40**, а `precision` удержался на уровне **0.61** (против 0.28 при балансировке).
>
> * **Сохранение лидерства на крупных классах:**
>   * Популярные кухни (*mexican*, *italian*, *chinese*) сохранили пиковые метрики.
>   * Для *chinese* F1-score вырос с **0.81** до **0.82**.
